In [17]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Markdown, Javascript
import base64
import io
import sympy as sp
import json

%matplotlib inline

# =====================================================================
# PARÁMETROS BASE Y VARIABLES SIMBÓLICAS
# =====================================================================
aranceles_rango = np.linspace(0, 200, 300)
subsidios = np.linspace(0, 200, 300)
P = 300

estado_simulador = {}
t_sym, F_sym, s_sym = sp.symbols('t F s')

# =====================================================================
# ESCENARIOS Y SUS DESCRIPCIONES
# =====================================================================
explicaciones = {
    'Personalizado (Modo Libre)':
        "Mueve los controles libremente o altera las funciones matemáticas base para crear tus propias hipótesis de mercado. Observa cómo cambiar las tasas altera la inclinación de las flechas (velocidades de daño) y cómo el límite fiscal se adapta dinámicamente a la nueva curva de costos del Estado. Aprovecha esta flexibilidad para evaluar si tus políticas públicas lograrían mitigar el impacto negativo en la economía local.",

    'Libre Mercado (Óptimo de Pareto)':
        "Condiciones estándar sin intervenciones. Al no existir aranceles externos ni retaliaciones, el Estado no necesita gastar dinero en subsidios compensatorios. El bienestar social se encuentra en su punto de equilibrio natural, maximizando la eficiencia económica. Es el modelo de referencia para comparar cuánto valor destruye una guerra comercial frente a un escenario de total cooperación internacional.",

    'Guerra Comercial Total (Efecto Búmeran)':
        "Economía frágil y Estado ineficiente. El costo fiscal es muy alto y las industrias son altamente sensibles. Nuestra fuerte retaliación multiplica el daño interno, colapsando el agro y forzando al Estado a gastar excesivamente sin lograr mitigar el impacto real. La estrategia de castigo recíproco afecta severamente a ambas naciones y el bienestar general cae en picada por el cierre mutuo de fronteras.",

    'Crisis Estructural (Colapso Agrícola)':
        "El agro es extremadamente sensible a los aranceles. El rival ataca fuertemente y, al chocar con nuestro estrecho límite de caja (presupuesto bajo), el sector colapsa dramáticamente. Este caso ilustra el peligro de tener un sector primario altamente dependiente del exterior sin el respaldo financiero necesario; el Estado se ve atado de manos y el mercado sufre pérdidas irrecuperables a un ritmo alarmante.",

    'Resiliencia Tecnológica (Shock Asimétrico)':
        "El sector tecnológico es casi inmune. El ataque extranjero destruye severamente las exportaciones del agro, pero los servicios digitales y la robustez tecnológica logran sostener la mayor parte de la economía. Esto muestra la ventaja estratégica de diversificar la economía hacia servicios de alto valor agregado: mientras el mundo físico impone barreras, la economía digital permite amortiguar la caída del PIB nacional.",

    'Proteccionismo Unilateral (Absorción del Golpe)':
        "El extranjero ataca con fuerza, pero decidimos no responder. Mantenemos las funciones base intactas. Aunque parece una derrota diplomática, es una táctica para evitar la escalada. Requiere un esfuerzo fiscal gigantesco para rescatar al agro, pero protege a nuestro sector tecnológico de represalias secundarias, evitando que la guerra comercial contamine a las industrias más rentables del país.",

    'Escudo Fiscal Perfecto (Optimización Absoluta)':
        "Estado hiper-eficiente con alta liquidez. El costo de subsidiar es extremadamente bajo y los fondos soberanos abundan. Esto representa el poder de una economía con fuertes reservas fiscales; el Estado logra absorber todo el golpe comercial mediante subsidios eficientes, permitiendo inyectar capital hasta ubicar el punto exactamente en la cima del bienestar y aislando a las empresas del conflicto internacional."
}

# =====================================================================
# ESTILOS CSS Y TEXTOS INTRODUCTORIOS
# =====================================================================
estilo_forzado = widgets.HTML(value="""
<style>
    .simu-contenedor, .simu-contenedor * {
        color: #2C3E50 !important;
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif !important;
    }
    .simu-contenedor .widget-label { color: #0A2540 !important; font-weight: 600 !important; }
    .simu-contenedor .widget-readout { background: #F1F3F5 !important; border-radius: 4px; padding: 2px 6px !important; }

    .btn-pdf {
        background-color: #1A252C !important;
        color: white !important;
        font-weight: bold !important;
        font-size: 14px !important;
        border-radius: 6px !important;
        border: none !important;
    }
    .btn-pdf:hover { background-color: #2C3E50 !important; }

    .intro-box {
        max-width: 950px;
        margin: 0 auto 25px auto;
        background-color: #F8FAFC;
        border: 1px solid #E2E8F0;
        border-radius: 8px;
        padding: 20px 30px;
        text-align: justify;
    }
    .intro-title {
        color: #0A2540;
        font-size: 15.5px;
        font-weight: bold;
        margin-top: 15px;
        margin-bottom: 8px;
        border-bottom: 1px solid #CBD5E1;
        padding-bottom: 4px;
    }
    .intro-title:first-child { margin-top: 0; }
    .intro-text { color: #34495E; font-size: 14.5px; line-height: 1.6; margin-bottom: 0; }

    .teclado-btn { min-width: 35px !important; width: 45px !important; margin: 2px !important; font-family: monospace !important; font-weight: bold !important; font-size: 15px !important;}

    .card-diagnostico {
        width: 90% !important;
        background-color: #FFFFFF !important;
        border: 1px solid #E2E8F0 !important;
        border-radius: 8px !important;
        padding: 25px !important;
        box-shadow: 0px 4px 12px rgba(0,0,0,0.03) !important;
        margin: 20px auto !important;
        display: flex !important;
        flex-direction: column !important;
        align-items: center !important;
    }
</style>
""")

titulo_html = widgets.HTML("<h2 style='text-align: center; color: #0A2540; margin-bottom: 15px; font-weight: 700;'>Simulador de Conflicto Arancelario</h2>")

texto_introductorio = widgets.HTML(value="""
<div class='intro-box'>
    <div class='intro-title'>¿Qué es este simulador?</div>
    <p class='intro-text' style='margin-bottom: 15px;'>
        Esta herramienta interactiva modela en tiempo real el impacto macroeconómico de una guerra arancelaria. Te permite asumir el rol de un analista estatal que debe tomar decisiones críticas frente a los ataques comerciales de un país extranjero, buscando equilibrar la protección de la industria local con la responsabilidad fiscal.
    </p>
    <div class='intro-title'>¿Cómo utilizarlo?</div>
    <p class='intro-text' style='margin-bottom: 15px;'>
        A la izquierda encontrarás parámetros dinámicos (controles deslizantes) que representan tus herramientas de política pública: puedes modificar la agresividad del ataque externo, decidir el nivel de tu propia retaliación y ajustar el presupuesto fiscal disponible para rescatar a tus industrias estratégicas (Agro y Tecnología).
    </p>
    <div class='intro-title'>Modo Avanzado y Escenarios</div>
    <p class='intro-text'>
        Si deseas profundizar en la teoría económica, puedes editar directamente las funciones matemáticas en las cajas de texto para moldear la sensibilidad de los sectores o la eficiencia del gasto del Estado (puedes apoyarte en el Teclado Matemático desplegable). También puedes seleccionar un escenario de estudio a la derecha para cargar situaciones hipotéticas y ver cómo las gráficas, el bienestar social y las derivadas algebraicas se calculan al instante.
    </p>
</div>
""")

# =====================================================================
# INTERFAZ (WIDGETS) Y TECLADO MATEMÁTICO LIMPIO
# =====================================================================
layout_sl = widgets.Layout(width='100%', margin='0 0 15px 0')

func_agro_input = widgets.Text(value='1000 - 0.05 · F · t²', description='F. Agro (Q):', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%'), continuous_update=False)
func_tech_input = widgets.Text(value='500 - 4 · F · t', description='F. Tecno (Q):', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%'), continuous_update=False)
func_costo_input = widgets.Text(value='15 · s²', description='F. Costo (C):', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%'), continuous_update=False)

teclado_destino = widgets.Dropdown(options=['F. Agro (Q)', 'F. Tecno (Q)', 'F. Costo (C)'], value='F. Agro (Q)', description='Destino:', layout=widgets.Layout(width='200px', margin='0 0 10px 0'))

teclado_r1 = ['t', 'F', 's', '+', '-', '·', '²', '^', '(', ')']
teclado_r2 = ['7', '8', '9', '4', '5', '6', '1', '2', '3', '0', '.', 'DEL']

def on_teclado_click(b):
    target = teclado_destino.value
    w = None
    if target == 'F. Agro (Q)': w = func_agro_input
    elif target == 'F. Tecno (Q)': w = func_tech_input
    elif target == 'F. Costo (C)': w = func_costo_input

    if w:
        if b.description == 'DEL':
            w.value = w.value[:-1]
        else:
            w.value += b.description

botones_r1 = [widgets.Button(description=s) for s in teclado_r1]
botones_r2 = [widgets.Button(description=s) for s in teclado_r2]

for btn in botones_r1 + botones_r2:
    btn.add_class('teclado-btn')
    btn.on_click(on_teclado_click)

caja_r1 = widgets.HBox(botones_r1, layout=widgets.Layout(justify_content='center'))
caja_r2 = widgets.HBox(botones_r2, layout=widgets.Layout(justify_content='center'))
panel_teclado_interno = widgets.VBox([teclado_destino, caja_r1, caja_r2], layout=widgets.Layout(padding='10px', align_items='center'))

acordeon_teclado = widgets.Accordion(children=[panel_teclado_interno], layout=widgets.Layout(width='95%', margin='0 0 15px 0'))
acordeon_teclado.set_title(0, 'Teclado Matemático')
acordeon_teclado.selected_index = None

panel_funciones = widgets.VBox([func_agro_input, func_tech_input, func_costo_input, acordeon_teclado])

arancel_ext_slider = widgets.IntSlider(min=0, max=200, step=1, value=40, description='Ataque Ext (%):', style={'description_width': 'initial'}, layout=layout_sl)
retaliacion_col_slider = widgets.IntSlider(min=0, max=200, step=1, value=0, description='Retaliación (%):', style={'description_width': 'initial'}, layout=layout_sl)
presupuesto_slider = widgets.IntSlider(min=1000, max=15000, step=500, value=6000, description='Presupuesto ($):', style={'description_width': 'initial'}, layout=layout_sl)

panel_izquierdo = widgets.VBox([panel_funciones, arancel_ext_slider, retaliacion_col_slider, presupuesto_slider], layout=widgets.Layout(width='48%'))

# =====================================================================
# GUÍA COMPACTADA EN PANEL DERECHO
# =====================================================================
texto_ejemplos_funciones = widgets.HTML(value="""
<div style='background-color: #FFFFFF; border: 1px solid #CBD5E1; border-radius: 6px; padding: 10px; margin: 10px 0 0 0; font-size: 12.5px; color: #475569; line-height: 1.4;'>
    <strong style='color: #0A2540;'>Guía y formato de funciones admitidas:</strong><br>
    Las funciones se construyen definiendo parámetros independientes (constantes numéricas) que dan la forma a la curva, y variables dependientes del simulador:<br><br>
    • F. Agro (Q): Estructura general: <code style='background: #E2E8F0; padding: 1px 4px; border-radius: 3px;'>a - b · F · t²</code> <br> &nbsp;&nbsp;Ej. (Base): <code style='background: #E2E8F0; padding: 1px 4px; border-radius: 3px;'>1000 - 0.05 · F · t²</code> | Ej. (Crisis): <code style='background: #E2E8F0; padding: 1px 4px; border-radius: 3px;'>1200 - 0.15 · F · t²</code><br>
    • F. Tecno (Q): Estructura general: <code style='background: #E2E8F0; padding: 1px 4px; border-radius: 3px;'>c - d · F · t</code> <br> &nbsp;&nbsp;Ej. (Base): <code style='background: #E2E8F0; padding: 1px 4px; border-radius: 3px;'>500 - 4 · F · t</code> | Ej. (Resiliencia): <code style='background: #E2E8F0; padding: 1px 4px; border-radius: 3px;'>800 - 0.5 · F · t</code><br>
    • F. Costo (C): Estructura general: <code style='background: #E2E8F0; padding: 1px 4px; border-radius: 3px;'>k · s²</code> <br> &nbsp;&nbsp;Ej. (Base): <code style='background: #E2E8F0; padding: 1px 4px; border-radius: 3px;'>15 · s²</code> | Ej. (Escudo Perfecto): <code style='background: #E2E8F0; padding: 1px 4px; border-radius: 3px;'>5 · s²</code><br>
    <span style='color: #E11D48; font-weight: 600; display:block; margin-top: 6px;'>Atención: Reemplaza las letras constantes (a,b,c,d,k) por números. Usa estrictamente las variables del modelo (t: arancel externo, F: factor de guerra, s: subsidio).</span>
</div>
""")

casos_dropdown = widgets.Dropdown(
    options=list(explicaciones.keys()), value='Personalizado (Modo Libre)',
    description='Escenario:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', margin='0 0 15px 0')
)

texto_explicacion_widget = widgets.HTML(value="")

panel_derecho = widgets.VBox(
    [casos_dropdown, texto_explicacion_widget, texto_ejemplos_funciones],
    layout=widgets.Layout(width='48%', border='1px solid #E2E8F0', padding='15px 20px', background_color='#F8FAFC', border_radius='8px')
)

fila_central = widgets.HBox([panel_izquierdo, panel_derecho], layout=widgets.Layout(justify_content='space-between', align_items='center', margin='0 0 25px 0'))

salida_graficas = widgets.Output(layout=widgets.Layout(display='flex', justify_content='center'))

alerta_output = widgets.HTML(value="", layout=widgets.Layout(width='100%'))

boton_pdf = widgets.Button(description="Descargar Reporte PDF Completo", layout=widgets.Layout(width='320px', height='45px', margin='25px auto 5px auto'))
boton_pdf.add_class('btn-pdf')

salida_reporte = widgets.HTMLMath(layout=widgets.Layout(width='100%'))

panel_diagnostico = widgets.VBox([salida_reporte, boton_pdf], layout=widgets.Layout(width='100%'))
panel_diagnostico.add_class('card-diagnostico')

salida_matematicas = widgets.Output(layout=widgets.Layout(width='90%', margin='15px auto', padding='20px', border='1px solid #CBD5E1'))

salida_js = widgets.Output(layout=widgets.Layout(display='none'))

ui_completa = widgets.VBox([
    estilo_forzado, titulo_html, texto_introductorio, alerta_output, fila_central,
    salida_graficas, panel_diagnostico, salida_matematicas, salida_js
], layout=widgets.Layout(padding='30px', background_color='#FFFFFF', border='1px solid #E2E8F0', border_radius='12px'))
ui_completa.add_class("simu-contenedor")

# =====================================================================
# LÓGICA, ESCUDO PROTECTOR ACTUALIZADO Y PROCESAMIENTO
# =====================================================================

def limpiar_sintaxis(texto):
    t = texto.replace('·', '*')
    t = t.replace('×', '*')
    t = t.replace('²', '**2')
    t = t.replace('³', '**3')
    t = t.replace('^', '**')
    return t

def validar_y_obtener_expresion(texto, variables_permitidas, default_str):
    texto_limpio = limpiar_sintaxis(texto)
    caracteres_validos = set("0123456789.tFs+-*/() ")

    for char in texto_limpio:
        if char not in caracteres_validos:
            return sp.sympify(default_str), True

    try:
        expr = sp.sympify(texto_limpio)
        for sym in expr.free_symbols:
            if sym not in variables_permitidas:
                return sp.sympify(default_str), True
        return expr, False
    except Exception:
        return sp.sympify(default_str), True

def actualizar_simulador(*args):
    global estado_simulador

    arancel_extranjero = arancel_ext_slider.value
    arancel_colombia = retaliacion_col_slider.value
    presupuesto_max = presupuesto_slider.value
    caso_actual = casos_dropdown.value

    texto_explicacion_widget.value = f"""
    <strong style='color: #0A2540; font-size: 15px; display: block; margin-bottom: 8px; margin-top: 10px; border-bottom: 2px solid #CBD5E1; padding-bottom: 5px;'>
        Contexto Analítico:
    </strong>
    <p style='color: #2C3E50; font-size: 14.5px; line-height: 1.6; margin-top: 5px; text-align: justify;'>
        {explicaciones.get(caso_actual, '')}
    </p>
    """

    factor_guerra = 1.0 + (arancel_colombia * 0.005)

    expr_agro, err_agro = validar_y_obtener_expresion(func_agro_input.value, [t_sym, F_sym], '1000 - 0.05 * F * t**2')
    expr_tech, err_tech = validar_y_obtener_expresion(func_tech_input.value, [t_sym, F_sym], '500 - 4 * F * t')
    expr_costo, err_costo = validar_y_obtener_expresion(func_costo_input.value, [s_sym], '15 * s**2')

    mensajes_alerta = []

    if err_agro:
         mensajes_alerta.append("Atención - F. Agro: Entrada inválida o variable incorrecta detectada. Se usó la predeterminada.")
         func_agro_input.unobserve(evento_actualizar, names='value')
         func_agro_input.value = '1000 - 0.05 · F · t²'
         func_agro_input.observe(evento_actualizar, names='value')

    if err_tech:
         mensajes_alerta.append("Atención - F. Tecno: Entrada inválida o variable incorrecta detectada. Se usó la predeterminada.")
         func_tech_input.unobserve(evento_actualizar, names='value')
         func_tech_input.value = '500 - 4 · F · t'
         func_tech_input.observe(evento_actualizar, names='value')

    if err_costo:
         mensajes_alerta.append("Atención - F. Costo: Entrada inválida o variable incorrecta detectada. Se usó la predeterminada.")
         func_costo_input.unobserve(evento_actualizar, names='value')
         func_costo_input.value = '15 · s²'
         func_costo_input.observe(evento_actualizar, names='value')

    if mensajes_alerta:
        alerta_output.value = f"""
        <div style="max-width: 950px; margin: 0 auto 20px auto; text-align: left; background-color: #FFF1F2; border: 1px solid #FDA4AF; border-radius: 6px; padding: 15px; font-size: 13.5px; color: #9F1239; font-weight: bold; line-height: 1.6;">
            {"<br>".join(mensajes_alerta)}
        </div>
        """
    else:
        alerta_output.value = ""

    der_agro_sym = sp.diff(expr_agro, t_sym)
    der_tech_sym = sp.diff(expr_tech, t_sym)
    der_costo_sym = sp.diff(expr_costo, s_sym)

    f_agro = sp.lambdify((t_sym, F_sym), expr_agro, modules=['numpy'])
    f_tech = sp.lambdify((t_sym, F_sym), expr_tech, modules=['numpy'])
    f_costo = sp.lambdify(s_sym, expr_costo, modules=['numpy'])
    f_der_agro = sp.lambdify((t_sym, F_sym), der_agro_sym, modules=['numpy'])
    f_der_tech = sp.lambdify((t_sym, F_sym), der_tech_sym, modules=['numpy'])
    f_der_costo = sp.lambdify(s_sym, der_costo_sym, modules=['numpy'])

    Ingreso_agri_total = np.broadcast_to(f_agro(aranceles_rango, factor_guerra), aranceles_rango.shape)
    Ingreso_tech_total = np.broadcast_to(f_tech(aranceles_rango, factor_guerra), aranceles_rango.shape)

    Ingreso_agri_actual = float(f_agro(arancel_extranjero, factor_guerra))
    Ingreso_tech_actual = float(f_tech(arancel_extranjero, factor_guerra))

    derivada_agri = float(f_der_agro(arancel_extranjero, factor_guerra)) if Ingreso_agri_actual > -1500 else 0
    derivada_tech = float(f_der_tech(arancel_extranjero, factor_guerra)) if Ingreso_tech_actual > -1500 else 0

    arancel_efectivo = np.maximum(0, arancel_extranjero - subsidios)
    Ingreso_agri_mitigado = np.broadcast_to(f_agro(arancel_efectivo, factor_guerra), subsidios.shape)

    Costo_Fiscal_Rango = np.broadcast_to(f_costo(subsidios), subsidios.shape)
    Recaudo_Retaliacion = (arancel_colombia * 150)
    Bienestar_Escudo = (P * Ingreso_agri_mitigado) - Costo_Fiscal_Rango + Recaudo_Retaliacion

    excede_presupuesto = Costo_Fiscal_Rango > presupuesto_max
    if np.any(excede_presupuesto):
        idx_limite = np.argmax(excede_presupuesto)
        subsidio_max_pagable = subsidios[idx_limite]
    else:
        subsidio_max_pagable = 200

    indice_max_teorico = np.argmax(Bienestar_Escudo)
    subsidio_optimo_teorico = subsidios[indice_max_teorico]

    techo_alcanzado = subsidio_optimo_teorico > subsidio_max_pagable
    subsidio_desplegado = subsidio_max_pagable if techo_alcanzado else subsidio_optimo_teorico

    idx_ejecutado = (np.abs(subsidios - subsidio_desplegado)).argmin()
    bienestar_realizado = Bienestar_Escudo[idx_ejecutado]
    costo_realizado = Costo_Fiscal_Rango[idx_ejecutado]

    derivada_costo_fiscal = float(f_der_costo(subsidio_desplegado))
    arancel_efec_actual = max(0, arancel_extranjero - subsidio_desplegado)

    derivada_agri_en_efectivo = float(f_der_agro(arancel_efec_actual, factor_guerra))

    if arancel_efec_actual > 0:
        derivada_bienestar = P * (-derivada_agri_en_efectivo) - derivada_costo_fiscal
    else:
        derivada_bienestar = -derivada_costo_fiscal

    rec_retaliacion = ""
    if arancel_colombia == 0:
        rec_retaliacion = f"<b>Sobre la Retaliación:</b> Actualmente se mantiene una postura de absorción pasiva ({arancel_colombia}% retaliación). Económicamente es la decisión más prudente para blindar la producción local. Puesto que la fórmula del Factor de Guerra amplifica las sensibilidades de las gráficas, responder al fuego extranjero solo aceleraría la destrucción de volumen de exportación tanto en tecnología como en agro. Se recomienda <b>mantener esta postura neutra</b> a menos que consideraciones geopolíticas forzosas exijan recaudar aranceles."
    elif arancel_colombia <= arancel_extranjero * 0.5:
        rec_retaliacion = f"<b>Sobre la Retaliación:</b> Existe una retaliación moderada ({arancel_colombia}%). Si bien esto provee un ingreso a las arcas estatales, incrementa el multiplicador de fricción (F = {factor_guerra:.2f}). Matemáticamente, el daño de distorsión interna supera el recaudo obtenido. Se recomienda <b>reducir gradualmente este arancel</b> para aliviar la presión sobre las curvas de oferta locales."
    else:
        rec_retaliacion = f"<b>Sobre la Retaliación:</b> La nación sostiene una guerra comercial severa ({arancel_colombia}% de retaliación), resultando en un factor de guerra de F = {factor_guerra:.2f}. Esta táctica recíproca está hundiendo rápidamente a las industrias exportadoras. Se recomienda <b>reducir drásticamente la retaliación</b>; el afán de castigar la economía del rival está causando un auto-boicot insostenible en el tejido productivo propio."

    rec_presupuesto = ""
    if techo_alcanzado:
        rec_presupuesto = f"<b>Sobre la Intervención y el Presupuesto:</b> El Estado enfrenta una severa <b>restricción de liquidez</b>. El subsidio ha chocado contra el tope del presupuesto (USD {presupuesto_max:,.0f}). Ya que la pendiente del bienestar sigue siendo positiva, esto significa que hay rentabilidad social sin explotar. Se recomienda urgentemente <b>ampliar el presupuesto fiscal</b> (o emitir deuda a corto plazo) para continuar subsidiando y alcanzar el punto de equilibrio que garantice el máximo bienestar social posible."
    elif derivada_bienestar > 0.5:
        rec_presupuesto = f"<b>Sobre la Intervención y el Presupuesto:</b> La política actual está <b>sub-optimizada</b> (Pendiente = +{derivada_bienestar:.1f}). Existe presupuesto ocioso y cada unidad extra inyectada como subsidio generará un valor social superior a su costo marginal. Se recomienda <b>incrementar la tasa de subsidios</b> para movernos hacia la derecha en la gráfica de optimización fiscal y maximizar el escudo protector sobre el sector productivo."
    elif derivada_bienestar < -0.5:
        rec_presupuesto = f"<b>Sobre la Intervención y el Presupuesto:</b> El Estado presenta una <b>sobre-protección altamente ineficiente</b> (Pendiente = {derivada_bienestar:.1f}). Costear el actual {subsidio_desplegado:.1f}% de subsidios está drenando recursos mucho más rápido del beneficio que otorga la recuperación del agro. Se recomienda <b>disminuir inmediatamente el gasto público</b> retrocediendo en los subsidios hasta equilibrar el costo marginal."
    else:
        rec_presupuesto = f"<b>Sobre la Intervención y el Presupuesto:</b> Excelente gestión fiscal. La política pública actual está <b>perfectamente calibrada y optimizada</b>. El subsidio de {subsidio_desplegado:.1f}% ubica al país justo en la cima del bienestar social, donde el costo marginal se equilibra con el beneficio. Se recomienda <b>mantener estrictamente este esquema de gasto</b> (USD {costo_realizado:,.0f}), asegurando la estabilidad macroeconómica."

    recomendacion_texto = f"{rec_retaliacion} <br><br> {rec_presupuesto}"

    with salida_graficas:
        clear_output(wait=True)
        plt.style.use('default')
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), facecolor='#FFFFFF')

        for ax in [ax1, ax2]:
            ax.set_facecolor('#FFFFFF')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.grid(color='#F1F3F5', linestyle='-', linewidth=0.8)

        ax1.fill_between(aranceles_rango, 0, -1500, color='#FADBD8', alpha=0.3, label='Zona de Pérdidas')
        ax1.axhline(y=0, color='#E74C3C', linestyle='-', linewidth=1.2, alpha=0.6)
        ax1.plot(aranceles_rango, Ingreso_agri_total, color='#27AE60', linewidth=2.5, label='Curva Agro')
        ax1.plot(aranceles_rango, Ingreso_tech_total, color='#2980B9', linewidth=2.5, label='Curva Tecnología')
        ax1.scatter(arancel_extranjero, Ingreso_agri_actual, color='#1E8449', s=130, zorder=5, edgecolor='white', linewidth=1.5,
                    label=f'Agro: {Ingreso_agri_actual:.0f} u. | dQ/d arancel: {derivada_agri:.1f}')
        ax1.scatter(arancel_extranjero, Ingreso_tech_actual, color='#1F618D', s=130, zorder=5, edgecolor='white', linewidth=1.5,
                    label=f'Tecno: {Ingreso_tech_actual:.0f} u. | dQ/d arancel: {derivada_tech:.1f}')

        delta_x = 15
        if arancel_extranjero < 185:
            ax1.annotate('', xy=(arancel_extranjero + delta_x, Ingreso_agri_actual + (delta_x * derivada_agri)),
                         xytext=(arancel_extranjero, Ingreso_agri_actual), zorder=4, arrowprops=dict(arrowstyle="->", lw=2, color="#1E8449"))
            ax1.annotate('', xy=(arancel_extranjero + delta_x, Ingreso_tech_actual + (delta_x * derivada_tech)),
                         xytext=(arancel_extranjero, Ingreso_tech_actual), zorder=4, arrowprops=dict(arrowstyle="->", lw=2, color="#1F618D"))

        ax1.axvline(x=arancel_extranjero, color='#95A5A6', linestyle='--', alpha=0.5)
        ax1.set_title('1. Oferta Exportable y Velocidad de Daño', fontsize=13, fontweight='bold', color='#0A2540', pad=15)
        ax1.set_xlabel('Arancel Externo Evaluado (%)', fontsize=10, color='#2C3E50')
        ax1.set_ylabel('Unidades Exportadas Netas', fontsize=10, color='#2C3E50')
        ax1.set_xlim(0, 200); ax1.set_ylim(-1500, 1500)
        ax1.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='#E2E8F0')

        ax2.fill_between(subsidios, 0, Bienestar_Escudo, color='#F1C40F', alpha=0.1)
        ax2.axvspan(subsidio_max_pagable, 200, color='#FADBD8', alpha=0.2, hatch='//', label='Zona Fiscal Restringida')
        ax2.plot(subsidios, Bienestar_Escudo, color='#F39C12', linewidth=2.5, label='Bienestar Social Potencial')
        ax2.axvline(x=subsidio_max_pagable, color='#C0392B', linestyle='-', linewidth=2, label=f'Límite Caja ({subsidio_max_pagable:.1f}%)')
        ax2.scatter(subsidio_desplegado, bienestar_realizado, color='#8E44AD', s=140, zorder=5, edgecolor='white', linewidth=1.5,
                    label=f'Bienestar Realizado: USD {bienestar_realizado:,.0f}')

        x_tan = np.linspace(max(0, subsidio_desplegado - 20), min(200, subsidio_desplegado + 20), 10)
        y_tan = bienestar_realizado + derivada_bienestar * (x_tan - subsidio_desplegado)
        ax2.plot(x_tan, y_tan, color='#8E44AD', linestyle='--', linewidth=2, alpha=0.7, label=f'Pendiente (dB/ds): {derivada_bienestar:.1f}')

        ax2.set_title('2. Optimización Fiscal y Bienestar Social', fontsize=13, fontweight='bold', color='#0A2540', pad=15)
        ax2.set_xlabel('Subsidio Estatal Inyectado (%)', fontsize=10, color='#2C3E50')
        ax2.set_ylabel('Bienestar Social (USD)', fontsize=10, color='#2C3E50')
        ax2.set_xlim(0, 200)

        max_b = max(Bienestar_Escudo) if max(Bienestar_Escudo) > 0 else 1000
        min_b = min(Bienestar_Escudo) if min(Bienestar_Escudo) < 0 else 0
        margen_y = abs(max_b * 0.2)
        ax2.set_ylim(min_b, max_b + margen_y)
        ax2.legend(loc='lower left', frameon=True, facecolor='white', edgecolor='#E2E8F0', fontsize=9)

        fig.subplots_adjust(left=0.08, right=0.92, bottom=0.12, top=0.88, wspace=0.25)

        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight', dpi=120)
        buf.seek(0)
        estado_simulador['img_base64'] = f"data:image/png;base64,{base64.b64encode(buf.read()).decode('utf-8')}"
        buf.close()

        plt.show()

    latex_agro = sp.latex(expr_agro)
    latex_tech = sp.latex(expr_tech)
    latex_costo = sp.latex(expr_costo)
    latex_der_agro = sp.latex(der_agro_sym)
    latex_der_costo = sp.latex(der_costo_sym)

    reporte_html = f"""
    <div style="width: 100%; text-align: center;">
        <h3 style="color: #0A2540; border-bottom: 2px solid #F1F3F5; padding-bottom: 12px; margin-top: 0; font-weight: 700;">DIAGNÓSTICO DEL MODELO ACTUAL</h3>

        <div style="display: flex; flex-wrap: wrap; justify-content: space-between; margin-top: 20px; text-align: left;">
            <div style="width: 30%; min-width: 220px; margin-bottom: 15px;">
                <strong style="color: #0A2540; font-size: 14.5px;">PARÁMETROS DE ENTRADA</strong>
                <ul style="color: #2C3E50; font-size: 13.5px; padding-left: 20px; line-height: 1.7; margin-top: 10px;">
                    <li>Ataque Externo: <b>{arancel_extranjero}%</b></li>
                    <li>Retaliación Local: <b>{arancel_colombia}%</b></li>
                    <li>Límite de Presupuesto: <b>USD {presupuesto_max:,.0f}</b></li>
                </ul>
            </div>

            <div style="width: 33%; min-width: 220px; margin-bottom: 15px;">
                <strong style="color: #0A2540; font-size: 14.5px;">FUNCIONES DEL SISTEMA</strong>
                <ul style="color: #2C3E50; font-size: 13.5px; padding-left: 20px; line-height: 1.7; margin-top: 10px;">
                    <li>F. Agro: <b>{func_agro_input.value}</b></li>
                    <li>F. Tecno: <b>{func_tech_input.value}</b></li>
                    <li>F. Costo: <b>{func_costo_input.value}</b></li>
                </ul>
            </div>

            <div style="width: 33%; min-width: 220px; margin-bottom: 15px; border-left: 2px dashed #E2E8F0; padding-left: 20px;">
                <strong style="color: #0A2540; font-size: 14.5px;">RESULTADOS CLAVE</strong>
                <ul style="color: #2C3E50; font-size: 13.5px; padding-left: 20px; line-height: 1.7; margin-top: 10px;">
                    <li>Subsidio Desplegado: <b>{subsidio_desplegado:.1f}%</b></li>
                    <li>Arancel Efectivo: <b>{arancel_efec_actual:.1f}%</b></li>
                    <li>Volumen Agro: <b>{Ingreso_agri_actual:.0f} u.</b></li>
                    <li>Volumen Tecno: <b>{Ingreso_tech_actual:.0f} u.</b></li>
                    <li>Costo Fiscal Ejecutado: <b>USD {costo_realizado:,.0f}</b></li>
                    <li style="color: #27AE60; font-weight: bold;">Bienestar Neto: USD {bienestar_realizado:,.0f}</li>
                </ul>
            </div>
        </div>

        <div style="background-color: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 6px; padding: 15px; margin-top: 10px; text-align: center;">
            <strong style="color: #0A2540; font-size: 14px;">VELOCIDADES (DERIVADAS EN EL PUNTO ACTUAL)</strong>
            <p style="color: #2C3E50; font-size: 13.5px; line-height: 1.6; margin-top: 8px; margin-bottom: 0;">
                Agro: pierde <b>{abs(derivada_agri):.1f}%</b> &nbsp;|&nbsp;
                Tecnología: pierde <b>{abs(derivada_tech):.1f}%</b> &nbsp;|&nbsp;
                Costo Fiscal: sube <b>USD {derivada_costo_fiscal:,.1f}</b> &nbsp;|&nbsp;
                Pendiente Bienestar Total: <b>{derivada_bienestar:.1f}</b>
            </p>
        </div>

        <div style="margin-top: 25px; background-color: #FFFFFF; padding: 20px; border-radius: 6px; border-left: 5px solid #2980B9; box-shadow: 0px 2px 8px rgba(0,0,0,0.05); text-align: left;">
            <strong style="color: #0A2540; display: block; font-size: 15.5px; margin-bottom: 12px;">RECOMENDACIONES TRAS ANÁLISIS:</strong>
            <p style="color: #2C3E50; text-align: justify; font-size: 14px; line-height: 1.6; margin-top: 0; margin-bottom: 0;">
                {recomendacion_texto}
            </p>
        </div>
    </div>
    """
    salida_reporte.value = reporte_html
    estado_simulador['diagnostico_html'] = reporte_html

    matematicas_md = f"""
<div style="padding: 10px;">
<h3 style="color: #0A2540; text-align: center; border-bottom: 2px solid #E2E8F0; padding-bottom: 10px; font-weight: 700;">PROCEDIMIENTO MATEMÁTICO: OPTIMIZACIÓN BASADA EN DERIVADAS</h3>

El simulador utiliza el cálculo diferencial para encontrar las razones de cambio (velocidades) tanto del daño económico como del costo fiscal. A continuación se detalla el procedimiento paso a paso:

#### PASO 1: Calcular el Acelerador por Retaliación ($F$)
El factor de guerra ($F$) amplifica el daño si el país decide contraatacar. Depende del arancel de retaliación impuesto por nosotros ($R$).
* **Función original:** $$ F(R) = 1 + (R \\cdot 0.005) $$
* **Evaluación ($R = {arancel_colombia}$):** $$ F = 1 + ({arancel_colombia} \\cdot 0.005) $$
* **Resultado final:** $$ F = {factor_guerra:.2f} $$

#### PASO 2: Razón de Cambio de Exportaciones Agropecuarias (Derivada)
Para medir qué tan rápido caen las exportaciones al subir el arancel, tomamos la función de producción y calculamos su derivada respecto al arancel ($t$).
* **Función original:** $$ Q_{{agri}}(t) = {latex_agro} $$
* **Cálculo de la derivada ($dQ/dt$):** $$ \\frac{{dQ_{{agri}}}}{{dt}} = {latex_der_agro} $$
* **Evaluación:** Reemplazamos $F = {factor_guerra:.2f}$ y el arancel efectivo evaluado $t = {arancel_efec_actual:.1f}$:
* **Resultado final:** $$ \\frac{{dQ_{{agri}}}}{{dt}} = {derivada_agri_en_efectivo:.2f} \\text{{ unidades / \\%}} $$
*(Una tasa negativa indica la velocidad a la que se pierden exportaciones en este punto).*

#### PASO 3: Razón de Cambio del Costo Fiscal (Derivada)
Para saber cuánto le cuesta al Estado aumentar un punto porcentual extra de subsidio ($s$), derivamos la función de costo fiscal.
* **Función original:** $$ C(s) = {latex_costo} $$
* **Cálculo de la derivada ($dC/ds$):** $$ \\frac{{dC}}{{ds}} = {latex_der_costo} $$
* **Evaluación:** Reemplazamos el nivel de subsidio actual $s = {subsidio_desplegado:.1f}$:
* **Resultado final:** $$ \\frac{{dC}}{{ds}} = {derivada_costo_fiscal:.2f} \\text{{ USD / \\%}} $$

#### PASO 4: Pendiente del Bienestar Social (Optimización)
El bienestar total del país ($B$) relaciona el valor de las exportaciones salvadas (multiplicadas por su precio $P$) frente al costo de los subsidios. Buscamos el punto donde la pendiente de esta curva sea cero.
* **Fórmula de la pendiente ($dB/ds$):** $$ \\frac{{dB}}{{ds}} = P \\cdot \\left(-\\frac{{dQ_{{agri}}}}{{dt}}\\right) - \\frac{{dC}}{{ds}} $$
* **Evaluación:** Usamos el precio constante $P = 300$ y las razones de cambio calculadas en los Pasos 2 y 3:
* **Operación aritmética:** $$ \\frac{{dB}}{{ds}} = 300 \\cdot \\left(-({derivada_agri_en_efectivo:.4f})\\right) - ({derivada_costo_fiscal:.2f}) $$
* **Resultado Final:** $$ \\frac{{dB}}{{ds}} = {derivada_bienestar:.2f} $$

> *Interpretación analítica: Si la pendiente es positiva, conviene aumentar el subsidio para ganar bienestar. Si es negativa, cuesta más de lo que beneficia y debe reducirse. Si es cero, se ha hallado el equilibrio óptimo perfecto.*
</div>
    """

    with salida_matematicas:
        clear_output(wait=True)
        display(Markdown(matematicas_md))

    procedimiento_pdf_html = f"""
    <div style="width: 100%; margin-top: 25px;">
        <h3 style="color: #0A2540; border-bottom: 2px solid #F1F3F5; padding-bottom: 12px; font-weight: 700; text-align: center;">PROCEDIMIENTO MATEMÁTICO: OPTIMIZACIÓN BASADA EN DERIVADAS</h3>

        <p style="font-size: 14.5px; color: #2C3E50; line-height: 1.6; text-align: justify; margin-bottom: 20px;">
            El simulador utiliza el cálculo diferencial para encontrar las razones de cambio (velocidades) tanto del daño económico como del costo fiscal. A continuación se detalla el paso a paso:
        </p>

        <div style="margin-bottom: 20px;">
            <strong style="color: #0A2540; font-size: 14.5px;">PASO 1: Calcular el Acelerador por Retaliación (F)</strong>
            <p style="font-size: 13.5px; color: #2C3E50; line-height: 1.6; margin-top: 5px;">
                El factor de guerra (F) amplifica el daño si el país decide contraatacar. Depende del arancel de retaliación impuesto por nosotros (R).<br>
                <b>Función original:</b> \\[ F(R) = 1 + (R \\cdot 0.005) \\]
                <b>Evaluación (R = {arancel_colombia}):</b> \\[ F = 1 + ({arancel_colombia} \\cdot 0.005) \\]
                <b>Resultado final:</b> \\[ F = {factor_guerra:.2f} \\]
            </p>
        </div>

        <div style="margin-bottom: 20px;">
            <strong style="color: #0A2540; font-size: 14.5px;">PASO 2: Razón de Cambio de Exportaciones Agropecuarias (Derivada)</strong>
            <p style="font-size: 13.5px; color: #2C3E50; line-height: 1.6; margin-top: 5px;">
                Para medir qué tan rápido caen las exportaciones al subir el arancel, tomamos la función de producción y calculamos su derivada respecto al arancel (t).<br>
                <b>Función original:</b> \\[ Q_{{agri}}(t) = {latex_agro} \\]
                <b>Cálculo de la derivada (dQ/dt):</b> \\[ \\frac{{dQ_{{agri}}}}{{dt}} = {latex_der_agro} \\]
                <b>Evaluación:</b> Reemplazamos F = {factor_guerra:.2f} y el arancel efectivo t = {arancel_efec_actual:.1f}:<br>
                <b>Resultado final:</b> \\[ \\frac{{dQ_{{agri}}}}{{dt}} = {derivada_agri_en_efectivo:.2f} \\text{{ unidades / \\%}} \\]
                <i>(Nota: Un valor negativo indica la velocidad a la que se pierden ventas).</i>
            </p>
        </div>

        <div style="margin-bottom: 20px;">
            <strong style="color: #0A2540; font-size: 14.5px;">PASO 3: Razón de Cambio del Costo Fiscal (Derivada)</strong>
            <p style="font-size: 13.5px; color: #2C3E50; line-height: 1.6; margin-top: 5px;">
                Para saber el costo marginal para el Estado por cada punto porcentual extra de subsidio (s), derivamos la función de costo fiscal.<br>
                <b>Función original:</b> \\[ C(s) = {latex_costo} \\]
                <b>Cálculo de la derivada (dC/ds):</b> \\[ \\frac{{dC}}{{ds}} = {latex_der_costo} \\]
                <b>Evaluación:</b> Reemplazamos el nivel de subsidio actual s = {subsidio_desplegado:.1f}:<br>
                <b>Resultado final:</b> \\[ \\frac{{dC}}{{ds}} = {derivada_costo_fiscal:.2f} \\text{{ USD / \\%}} \\]
            </p>
        </div>

        <div style="margin-bottom: 20px;">
            <strong style="color: #0A2540; font-size: 14.5px;">PASO 4: Pendiente del Bienestar Social (Optimización)</strong>
            <p style="font-size: 13.5px; color: #2C3E50; line-height: 1.6; margin-top: 5px;">
                El bienestar total del país relaciona el valor de las exportaciones salvadas frente al costo de los subsidios. Buscamos el punto donde la pendiente de esta curva sea cero (Óptimo).<br>
                <b>Fórmula de la pendiente (dB/ds):</b> \\[ \\frac{{dB}}{{ds}} = P \\cdot \\left(-\\frac{{dQ_{{agri}}}}{{dt}}\\right) - \\frac{{dC}}{{ds}} \\]
                <b>Evaluación:</b> Usamos el precio constante P = 300 y las razones de cambio calculadas en los Pasos 2 y 3:<br>
                <b>Operación aritmética:</b> \\[ \\frac{{dB}}{{ds}} = 300 \\cdot \\left(-({derivada_agri_en_efectivo:.4f})\\right) - ({derivada_costo_fiscal:.2f}) \\]
                <b>Resultado Final:</b> \\[ \\frac{{dB}}{{ds}} = {derivada_bienestar:.2f} \\]
                <br><i>Interpretación analítica: Si la pendiente es positiva, conviene aumentar el subsidio. Si es negativa, conviene reducirlo. Si es cero, el Estado ha encontrado el equilibrio perfecto.</i>
            </p>
        </div>
    </div>
    """
    estado_simulador['procedimiento_html'] = procedimiento_pdf_html

# =====================================================================
# SOLUCIÓN DE DESCARGA DIRECTA (SIN POLIFILL, SIN ALERTAS DE IMPRESIÓN)
# =====================================================================
def exportar_diagnostico_a_pdf(b):
    diag_seguro = json.dumps(estado_simulador.get('diagnostico_html', ''))
    img_segura = json.dumps(estado_simulador.get('img_base64', ''))
    proc_seguro = json.dumps(estado_simulador.get('procedimiento_html', ''))

    js_final = f"""
    var diagnosticoHtml = {diag_seguro};
    var base64Img = {img_segura};
    var procedimientoHtml = {proc_seguro};

    var contenidoHtml = `
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Reporte de Diagnóstico Completo</title>
        <script id="MathJax-script" async src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js"></script>
        <style>
            body {{ font-family: "Segoe UI", sans-serif; margin: 45px; color: #2C3E50; }}
            h2, h3 {{ color: #0A2540; text-align: center; font-weight: 700; }}
            .grafica-contenedor {{ text-align: center; margin: 30px auto; width: 100%; }}
            .grafica-img {{ max-width: 95%; border: 1px solid #E2E8F0; border-radius: 8px; }}

            /* Estilo para el botón de impresión manual */
            .btn-imprimir {{
                display: block;
                width: 250px;
                margin: 0 auto 30px auto;
                padding: 12px;
                background-color: #2980B9;
                color: white;
                border: none;
                border-radius: 6px;
                font-size: 16px;
                font-weight: bold;
                cursor: pointer;
                text-align: center;
                box-shadow: 0 4px 6px rgba(0,0,0,0.1);
            }}
            .btn-imprimir:hover {{ background-color: #1A5276; }}

            @media print {{
                .page-break {{ page-break-before: always; }}
                .no-print {{ display: none !important; }}
            }}
        </style>
    </head>
    <body>
        <button class="no-print btn-imprimir" onclick="window.print()">🖨️ Guardar como PDF</button>

        <h2>REPORTE ESTRATÉGICO DE CONFLICTO ARANCELARIO</h2><hr>
        ${{diagnosticoHtml}}
        ${{base64Img ? `<div class="page-break"></div><h3>VISUALIZACIÓN GRÁFICA</h3><hr><div class="grafica-contenedor"><img class="grafica-img" src="${{base64Img}}" /></div>` : ''}}
        ${{procedimientoHtml ? `<div class="page-break"></div>${{procedimientoHtml}}` : ''}}
    </body>
    </html>
    `;

    var blob = new Blob([contenidoHtml], {{type: "text/html;charset=utf-8"}});
    var url = URL.createObjectURL(blob);
    var a = document.createElement("a");
    a.href = url;
    a.download = "Reporte_Simulador.html";
    document.body.appendChild(a);
    a.click();
    document.body.removeChild(a);
    URL.revokeObjectURL(url);
    """

    with salida_js:
        clear_output(wait=True)
        display(Javascript(js_final))

boton_pdf.on_click(exportar_diagnostico_a_pdf)

# =====================================================================
# MANEJADORES DE EVENTOS Y VINCULACIÓN
# =====================================================================
def evento_casos(change):
    caso = change.new
    for w in [arancel_ext_slider, retaliacion_col_slider, presupuesto_slider, func_agro_input, func_tech_input, func_costo_input]:
        try: w.unobserve(evento_actualizar, names='value')
        except: pass

    func_agro_input.value = '1000 - 0.05 · F · t²'
    func_tech_input.value = '500 - 4 · F · t'
    func_costo_input.value = '15 · s²'

    if caso == 'Libre Mercado (Óptimo de Pareto)': arancel_ext_slider.value = 0; retaliacion_col_slider.value = 0; presupuesto_slider.value = 6000
    elif caso == 'Guerra Comercial Total (Efecto Búmeran)': func_agro_input.value = '1000 - 0.08 · F · t²'; func_tech_input.value = '500 - 6 · F · t'; func_costo_input.value = '20 · s²'; arancel_ext_slider.value = 150; retaliacion_col_slider.value = 150
    elif caso == 'Crisis Estructural (Colapso Agrícola)': func_agro_input.value = '1200 - 0.15 · F · t²'; arancel_ext_slider.value = 120; retaliacion_col_slider.value = 50; presupuesto_slider.value = 1000
    elif caso == 'Resiliencia Tecnológica (Shock Asimétrico)': func_tech_input.value = '800 - 0.5 · F · t'; arancel_ext_slider.value = 100; retaliacion_col_slider.value = 20; presupuesto_slider.value = 4000
    elif caso == 'Proteccionismo Unilateral (Absorción del Golpe)': arancel_ext_slider.value = 100; retaliacion_col_slider.value = 0; presupuesto_slider.value = 9000
    elif caso == 'Escudo Fiscal Perfecto (Optimización Absoluta)': func_costo_input.value = '5 · s²'; arancel_ext_slider.value = 80; retaliacion_col_slider.value = 10; presupuesto_slider.value = 15000

    for w in [arancel_ext_slider, retaliacion_col_slider, presupuesto_slider, func_agro_input, func_tech_input, func_costo_input]:
        w.observe(evento_actualizar, names='value')
    actualizar_simulador()

def evento_actualizar(change):
    if casos_dropdown.value != 'Personalizado (Modo Libre)':
        try: casos_dropdown.unobserve(evento_casos, names='value')
        except: pass
        casos_dropdown.value = 'Personalizado (Modo Libre)'
        casos_dropdown.observe(evento_casos, names='value')
    actualizar_simulador()

casos_dropdown.observe(evento_casos, names='value')
for w in [arancel_ext_slider, retaliacion_col_slider, presupuesto_slider, func_agro_input, func_tech_input, func_costo_input]:
    w.observe(evento_actualizar, names='value')

display(ui_completa)
actualizar_simulador()